In [19]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("emilyalsentzer/Bio_ClinicalBERT", device=device)

def embed_phrases(phrases: list[str], batch_size: int = 512) -> np.ndarray:
    return model.encode(
        phrases,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

CUDA available: True
Device: NVIDIA RTX PRO 6000 Blackwell Server Edition


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
df = pd.read_parquet("CLEANED_INPUTEVENTS.parquet")
df = df.dropna(subset=["Phrase"])

embeddings = embed_phrases(df["Phrase"].tolist())

# Bio_ClinicalBERT outputs 768 dims — reduce to 256 with PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=256)
embeddings_256 = pca.fit_transform(embeddings)
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.3f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/17419 [00:00<?, ?it/s]

Explained variance: 0.992


In [ ]:
df["Embedding_256"] = list(embeddings_256)  # each cell holds a (256,) array
df.to_parquet("EMBEDDED_INPUTEVENTS.parquet")

In [21]:
import tempfile, os
from sklearn.decomposition import IncrementalPCA
import pyarrow.parquet as pq
import pyarrow as pa

CHUNK_SIZE = 1000000
N_COMPONENTS = 256
TEMP_EMB_DIR = "temp_embeddings"
os.makedirs(TEMP_EMB_DIR, exist_ok=True)

df_full = pd.read_parquet("CLEANED_CHARTEVENTS.parquet")
df_full = df_full.dropna(subset=["Phrase"]).reset_index(drop=True)

print("Pass 1: encoding and fitting PCA...")
ipca = IncrementalPCA(n_components=N_COMPONENTS)

for i in range(35 * CHUNK_SIZE, len(df_full), CHUNK_SIZE):
    chunk = df_full.iloc[i:i+CHUNK_SIZE]
    emb = model.encode(chunk["Phrase"].tolist(), batch_size=512, normalize_embeddings=True, show_progress_bar=False)
    np.save(os.path.join(TEMP_EMB_DIR, f"emb_{i//CHUNK_SIZE:04d}.npy"), emb)
    ipca.partial_fit(emb)
    print(f"  chunk {i//CHUNK_SIZE + 1} encoded and fitted")

print(f"Explained variance: {ipca.explained_variance_ratio_.sum():.3f}")

Pass 1: encoding and fitting PCA...
  chunk 36 encoded and fitted
  chunk 37 encoded and fitted
  chunk 38 encoded and fitted
  chunk 39 encoded and fitted
  chunk 40 encoded and fitted
  chunk 41 encoded and fitted
  chunk 42 encoded and fitted
  chunk 43 encoded and fitted
  chunk 44 encoded and fitted
  chunk 45 encoded and fitted
  chunk 46 encoded and fitted
  chunk 47 encoded and fitted
  chunk 48 encoded and fitted
  chunk 49 encoded and fitted
  chunk 50 encoded and fitted
  chunk 51 encoded and fitted
  chunk 52 encoded and fitted
  chunk 53 encoded and fitted
Explained variance: 0.995


In [ ]:
print("Pass 2: transforming and writing...")

#for i in range(0, len(df_full), CHUNK_SIZE):
for i in range(35 * CHUNK_SIZE, 54 * CHUNK_SIZE, CHUNK_SIZE):
    chunk = df_full.iloc[i:i+CHUNK_SIZE].copy()
    emb_path = os.path.join(TEMP_EMB_DIR, f"emb_{i//CHUNK_SIZE:04d}.npy")
    emb = np.load(emb_path)
    emb_256 = ipca.transform(emb).astype("float32")

    chunk["Embedding_256"] = list(emb_256)

    parquet_path = f"parquet_chunks/EMBEDDED_CHARTEVENTS_{i//CHUNK_SIZE:04d}.parquet"
    chunk.to_parquet(parquet_path, index=False)

    os.remove(emb_path)
    print(f"  chunk {i//CHUNK_SIZE + 1}: {len(chunk):,} rows written")

# cleanup
print("Done.")

In [ ]:
df["Embedding_256"] = list(embeddings_256)  # each cell holds a (256,) array
df.to_parquet("EMBEDDED_CHARTEVENTS.parquet")